In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import viewer
import dianne
import pandas as pd
import matplotlib.colors as mcolors
dianne.setNotebookWidth(100)

In [ ]:
dataPath = '/projects/chuang-lab/USERS/domans/testing-PDXNet-DIANNE/data/'
# dataPath = '../../data/pdx-data/'
samples = ['110696-CORE', '110696-WSI']

F = 1
fname = f'img.data.uni2-{F}.h5ad'
idm = './identity-matrix.csv'
xenium_to_he_matrices = {sample: idm for sample in samples}
xenium_bundle_paths = {'110696-CORE': f'{dataPath}/XE-110696-CORE/', '110696-WSI': None}

secondary_images = {'110696-CORE': f'{dataPath}/XE-110696-CORE/morphology.ome.tif',
                    '110696-WSI': f'{dataPath}/110696-WSI/image.ome.tiff'}
secondary_matrices = xenium_to_he_matrices.copy()

species = pd.Categorical(pd.read_csv(f'{dataPath}/XE-110696-CORE/species.csv', header=None)[0])
method = pd.Categorical(pd.read_csv(f'{dataPath}/XE-110696-CORE/method.csv', header=None)[0])
cluster = pd.Categorical(pd.read_csv(f'{dataPath}/XE-110696-CORE/cluster.csv', header=None)[0].astype(str))

classifierPaths = 'classifiers/'
dianne.setupClassifierPaths(classifierPaths)

patch_size = 6 # number of tiles, in each dimension, to include in a patch (e.g. 8 means 8x8=64 tiles per patch)
ts, mpp, tile_size = dianne.loadSTQParams(dataPath + samples[0], F)
ads, imgs, patchCoordinates, patchesCDFs, qs, ts, mpp, L, N = dianne.loadDataAndPreparePatches(samples, dataPath, fname, L=None, ts=ts, mpp=mpp, N=patch_size)
sizes = {s: ads[s].shape[0] for s in samples}
print(f'Prepared {patchesCDFs.shape[0]} patches')

In [ ]:
runfn = dianne.makeRunFn(patchCoordinates, ads, samples, qs, ts, mpp, tile_size=tile_size, patch_size=patch_size, PCMA_alpha=0.8, alpha_img=0.5, multiplier=2)
savefn = dianne.makeSaveFn(patchCoordinates, ads, samples, qs, ts, mpp, PCMA_alpha=0.8, tile_size=tile_size, patch_size=patch_size, body_overlap=0.25, classifierPaths=classifierPaths)
loadfn = dianne.makeLoadFn(classifierPaths)
listfn = dianne.makeListFn(classifierPaths)
used_labels = [species, method, cluster]
annotationsPalette = [{a: mcolors.to_hex(dianne.Set123(i)) for i, a in enumerate(labels.categories)} for labels in used_labels]
all_annotations = [{'110696-CORE': labels, '110696-WSI': None} for labels in used_labels]
drawings = viewer.create_viewer(samples, secondary_images, height="800px", run_inference_fn=runfn, sample_sizes=sizes,
                                xenium_mpp=0.2125, max_cells=10000, matrices=xenium_to_he_matrices, xenium_bundle_paths=xenium_bundle_paths,
                                secondary_images=imgs, secondary_matrices=secondary_matrices,
                                draw_on_secondary=True,
                                annotations=all_annotations, category_colors=annotationsPalette,
                                save_func=savefn, load_func=loadfn, list_names_func=listfn)[1]

In [ ]:
if False:
    clf = dianne.loadGUIClassifier(classifierPaths, 'some-name-gui')
else:
    # Train the classifier on the patches and the corresponding features
    body_overlap = 0.25 # Fraction overlap between the drawn annotations and the tiles
    clf, patchesCDFsMod, annotationsMod = dianne.getClassifierForFromStrokes(drawings, patchCoordinates, tile_size, body_overlap, patch_size,
                                                                            ads, samples, qs, augFunc=dianne.PCMA, alpha=0.8, seed=0, showPatches=False)
    # dianne.saveGUIClassifier(clf, classifierPaths, 'some-name-gui', samples, ts, mpp, patch_size, tile_size, body_overlap, qs, patchesCDFsMod, annotationsMod, drawings)

print(patchesCDFsMod.shape)

# Run inference and visualize the results
if True:
    if not clf is None:
        infSample = samples[0]
        multiplier = 2 # Interpolation parameter for the probability heatmap
        x, y, p = dianne.inferProbFast(ads[infSample], clf, qs, tsize=ts/mpp, R=2, erode=True)
        xi, yi, pi = dianne.interpolatePoints(x, y, p, multiplier=multiplier)
        # dianne.showProbImg(xi, yi, pi, f=2, figsize=(3, 3), ts=ts, mpp=mpp, title=infSample, invert=False)
        viewer.set_overlay_points(xi, yi, pi, infSample, delta=tile_size / multiplier, alpha=0.5, color_low='#FFA500', color_high='#0000FF')

In [ ]:
# Create probability masks, extract contours and visualize them on the H&E image
downsampled_map, fshape = dianne.makeProbMask(ads[infSample], imgs[infSample], x, y, p, ts=ts, mpp=mpp, downfactor=16, verbose=True)
geojson = dianne.extractContoursForQuPath(downsampled_map, fshape, cutoff=0.5, min_area=10**6, downfactor=16, sigma=100)
dianne.viewContoursOnImage(imgs[infSample], geojson, fshape, level=2, figsize=(12, 12), linewidth=1)